# StructureDCA basics

This tutorial covers the basic usage of StructureDCA, demonstrates its application to mutation effect prediction, and explains how to adjust key input parameters (distance cutoff, pLDDT filter, minimum sequence identity filter, and the option to consider the gap character)

In [ ]:
# Imports
import pandas as pd
import scipy.stats as st
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from structuredca import StructureDCA

In [ ]:
# In this tutorial, as an example we will consider stability change assay from the MegaScale (also included in ProteinGym) dataset on the MBD domain:
msa_path = '../test_data/6acv_A_29-94.fasta'
pdb_path = '../test_data/6acv_A_29-94.pdb'
dms_path = '../test_data/6acv_A_29-94.csv'
chain = 'A'

# Basic StructureDCA initialization

Let's see first how to initialize StructureDCA with default parameters.  
This step aligns the PDB structure to the target sequence, extracts the contact map with a distance cutoff of $8$ Angström, and builds a sparse DCA model using sequences with at least $25\%$ sequence identity to the target sequence.  
It returns a StructureDCA object which can then be used for downstream calculations.

In [ ]:
# Run StructureDCA
sdca = StructureDCA(
    msa_path = msa_path,
    pdb_path = pdb_path,
    chains = chain,    
)

In [ ]:
# Plotting the Frobenius norms of the inferred couplings shows that couplings were inferred only for residue pairs in contact.
frobenius_norm: np.ndarray = sdca.dca_model.frobenius_norm()

plt.imshow(frobenius_norm, cmap='Blues')
plt.colorbar()
plt.xticks([])
plt.yticks([])
plt.title("Frobenius norms of couplings inferred by StructureDCA")
plt.show()

# Evaluating energy of sequences and effects of mutations

Now let us use StructureDCA to evaluate the "evolutionary energy" `E` of a sequence and the effect `dE` of a mutation on the protein and compare predictions with experimental data.  
We will use methods:
- `sdca.eval_mutation()`
- `sdca.eval_sequence()`
- `sdca.eval_mutations_table()`

In [ ]:
# Evaluate effect of 1 (single or multiple) mutation (reweighting by RSA or not)
mutations: list[str] = ['N40P', 'N40P:Q43F']
for mut_str in mutations:
    sdcarsa_dE = sdca.eval_mutation(mut_str, reweight_by_rsa=True)
    sdca_dE = sdca.eval_mutation(mut_str, reweight_by_rsa=False)
    print(f"StructureDCA[RSA] score for mutation '{mut_str}' : {sdcarsa_dE:.4f}")
    print(f"StructureDCA      score for mutation '{mut_str}' : {sdca_dE:.4f}")

In [ ]:
# This is equivalent to computing the difference in energy between the mutant and the wild-type sequence

# Target sequence
target_sequence = sdca.target_sequence.sequence
E_target = sdca.eval_sequence(target_sequence, reweight_by_rsa=False)
print(f"Wild-type sequence : {target_sequence} -> E = {E_target:.4f}")

# Mutate target sequence with mutation 'N40P:Q43F'
mutant_sequence = target_sequence[:39] + 'P' + target_sequence[40:42] + 'F' + target_sequence[43:]
E_mutant = sdca.eval_sequence(mutant_sequence, reweight_by_rsa=False)
print(f"Mutant sequence    : {mutant_sequence} -> E = {E_mutant:.4f}")

print(f"dE['{mutations[-1]}'] = {E_mutant - E_target:.4f}")

We can evaluate effects `dE` of all single-site mutations on the protein with method `sdca.eval_mutations_table()`.  
This method returns a list that provides, for each mutation:
- mutation, with residues indexed according to the FASTA sequence numbering
- mutation, with residues indexed according to the PDB file numbering
- StructurDCA score
- StructurDCA score reweighted by RSA
- RSA list of the mutated residues
- pLDDT list of the mutated residues
- gap_ratio list at the mutated positions in the MSA
- warnings tags associated with this mutation predictions (such as missing residues in 3D structure)

In [ ]:
# Eval all single-site mutation with StructureDCA
mutations_table = sdca.eval_mutations_table(save_path="./tmp.csv")
mutations_table

In [ ]:
# For readability and downstream analysis the list can also be directly converted to a pandas DataFrame:
df_muts = pd.DataFrame(mutations_table)
df_muts

Now, let us consider a specific dataset of mutations

In [ ]:
# Read DMS dataset
df_dms = pd.read_csv(dms_path)
df_dms

We can use the method `sdca.eval_mutations_table()` by explicitly providing a list of mutations (as strings like `A54H` or `M13A:G15K`).  
We can also directly save the output to a CSV file using the `save_path` option.

In [ ]:
# Eval dataset mutations with StructureDCA
mutation_list: list[str] = list(df_dms['mutation'])
mutations_table = sdca.eval_mutations_table(
    mutation_list,
    save_path='./0_my_mutation_table.csv'
)
df_muts = pd.DataFrame(mutations_table)
df_muts

In [ ]:
# StructureDCA scores correlate well with the DMS experimental values, further improved by RSA reweighting (relevant for stability prediction):
print(f"Spearman(StructureDCA, DMS)= {st.spearmanr(df_muts['StructureDCA'], df_dms['DDG']).statistic:.2f}")
print(f"Spearman(RSA*StructureDCA, DMS)= {st.spearmanr(df_muts['RSA*StructureDCA'], df_dms['DDG']).statistic:.2f}")

You can also evaluate the effect of a mutation starting from an **alternative background sequence** (other than the default target sequence, which was the first sequence in the MSA).

In [ ]:
# Define an alternative background sequence
seq1 = "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA" # background sequence
seq2 = "AACCCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA" # mutated background sequence
mut = "A3C:A4C:A5C" # mutation from seq1 -> seq2

# Evaluate envolutionary energies of seq1, seq2 and the difference
E1 = sdca.eval_sequence(seq1)
E2 = sdca.eval_sequence(seq2)
print(f"Alternative background sequence         (seq1) : {seq1} -> E = {E1:.4f}")
print(f"Mutated alternative background sequence (seq2) : {seq2} -> E = {E2:.4f}")
print(f"dE[seq1 -> seq2] = {E2 - E1:.4f}")

# Evaluate dE from seq1 -> seq2
dE = sdca.eval_mutation(mut, background_sequence=seq1)
print(f"dE[{mut}]  = {dE:.4f}")

# Optional arguments

## Distance cutoff

By default, StructureDCA uses a threshold of $8$ Angström to classify a residue pair as "in contact".  
This threshold can be set to a different value.

In [ ]:
# Run StructureDCA with d=15.0
sdca_d_high = StructureDCA(
    msa_path = msa_path,
    pdb_path = pdb_path,
    chains = chain,
    distance_cutoff = 15.0
)

# Plotting the Frobenius norms of the inferred couplings with d=15.0 show that more couplings here inferred (then with default d=8.0)
plt.imshow(sdca_d_high.dca_model.frobenius_norm(), cmap='Blues')
plt.colorbar()
plt.xticks([])
plt.yticks([])
plt.title("Frobenius norms of couplings inferred by StructureDCA (d=15)")
plt.show()

In [ ]:
# Run StructureDCA with d=5.0
sdca_d_low = StructureDCA(
    msa_path = msa_path,
    pdb_path = pdb_path,
    chains = chain,
    distance_cutoff = 5.0
)

# Plotting the Frobenius norms of the inferred couplings with d=5.0 show that less couplings here inferred (then with default d=8.0)
plt.imshow(sdca_d_low.dca_model.frobenius_norm(), cmap='Blues')
plt.colorbar()
plt.xticks([])
plt.yticks([])
plt.title("Frobenius norms of couplings inferred by StructureDCA (d=5)")
plt.show()

## pLDDT filter

In AlphaFold-predicted structures, the 3D Structure file includes a per-residue pLDDT score that reflects local confidence in the predicted structure.  
StructureDCA provides the option `use_contacts_plddt_filter` (disabeled by default) to exclude residue pairs involving low-confidence regions from being classified as contacts, to prevent artifactual contact assignments.  

By default, contacts in regions with a pLDDT lower than $70$ are excluded, except in a diagonal window of $1$ residue. These values can be changed with the `contacts_plddt_cutoff` and `contacts_plddt_keep_window` parameters.

In [ ]:
sdca_plddt_filter = StructureDCA(
    msa_path = msa_path,
    pdb_path = pdb_path,
    chains = chain,
    use_contacts_plddt_filter = True,
    contacts_plddt_cutoff = 70.0,
    contacts_plddt_keep_window = 1,
)

# The resulting coupling matrix is sparser and a few contacts were excluded with respect to the previous StructureDCA run.
plt.imshow(sdca_plddt_filter.dca_model.frobenius_norm(), cmap='Blues')
plt.colorbar()
plt.xticks([])
plt.yticks([])
plt.title("Frobenius norms of couplings inferred by StructureDCA with a PLDDT filter")
plt.show()

In [ ]:
# Plot removed contacts by the pLDDT filter
plt.imshow(sdca.contacts ^ sdca_plddt_filter.contacts, cmap='Greys')
plt.xticks([])
plt.yticks([])
plt.title("Contacts removed by the pLDDT filter")
plt.show()

In [ ]:
# These excluded contacts do correspond to regions with lower pLDDT values:
# We see a low pLDDT region in the middle of the protien sequence (aroung residue 20) and in the right terminus of the protein
cmap_plddt = mcolors.ListedColormap([
    (1.000, 0.490, 0.271),
    (1.000, 0.859, 0.075),
    (0.396, 0.796, 0.953),
    (0.000, 0.325, 0.839),
])
norm_plddt = mcolors.BoundaryNorm([0, 50, 70, 90, 100], cmap_plddt.N)

plt.figure(figsize=(15, 1))
plt.imshow(
    sdca_plddt_filter.plddt_array.reshape(1, -1),
    cmap=cmap_plddt,
    norm=norm_plddt,
)
plt.colorbar()
n = len(sdca_plddt_filter.plddt_array)
positions = np.arange(0, n, 1)
plt.yticks([])
plt.xticks(positions, positions + 1, fontsize=6)
plt.xlabel('Residue position')
plt.show()

## Minimum sequence identity filter

By default, sequences in the MSA that share less than $25\%$ sequence identity with the target are filtered out, as our previous work showed that excluding sequences in the homology twilight zone improves mutation effect prediction. This threshold may be adjusted or disabled using the `min_seqid` parameter.

In [ ]:
sdca_no_seqid_filter = StructureDCA(
    msa_path = msa_path,
    pdb_path = pdb_path,
    chains = chain,
    min_seqid = 0.0
)

# We can check that that the MSA is indeed deeper (though in this small example the difference is not significant):
print("")
print(f"MSA depth with default StructureDCA: {sdca.N}")
print(f"MSA depth with min_seqid=0.0: {sdca_no_seqid_filter.N}")

## Consider the gap character as the 21st amino acid

By default, StructureDCA ignores gap characters in the MSA. This prevents highly gapped positions from biasing the model coefficients.  
However, in some cases you might want to score sequences that contain gaps, which is not possible with the default settings.

In [ ]:
# Evaluate sequences with gaps

# Target sequence
print(f"Target sequence: '{sdca.target_sequence.sequence}'")
print(f"Target sequence score: {sdca.eval_sequence(sdca.target_sequence.sequence):.2f}")
print('\n')

# Target sequence with a few gaps
target_sequence_with_gaps = '-' + sdca.target_sequence.sequence[1:-2] + '--'
print(f"Trying to score: '{target_sequence_with_gaps}'")

try:
    sdca.eval_sequence(target_sequence_with_gaps)
except ValueError as err:
    print(err)

In [ ]:
# This behavior can be modified by setting the `exclude_gaps` parameter to `False`.
sdca_with_gaps = StructureDCA(
    msa_path = msa_path,
    pdb_path = pdb_path,
    chains = chain,
    exclude_gaps = False
)

# Log results
print("")
print(f"Target sequence with gaps: '{target_sequence_with_gaps}'")
print(f"Gapped sequence score: {sdca_with_gaps.eval_sequence(target_sequence_with_gaps):.2f}")

# All StructureDCA arguments

Other relevant StructureDCA arguments that were not discussed in this tutorial are briefly commented below:

In [ ]:
sdca_all = StructureDCA(
    msa_path = msa_path, # path to MSA file (str)
    pdb_path = pdb_path, # path to PDB file (str)
    chains = chain,      # target chain(s)
    
    distance_cutoff = 8.00, # distance threshold (in Å) to consider a residue-residue contact

    # L2 regularization coefficients: lambda_x = lambda_x (1 + lambda_asymptotic * Neff)
    lambda_h = 1.00,                # coefficient for h
    lambda_J = 1.00,                # coefficient for J
    lambda_asymptotic = 0.001, # asymptotic correction (when Neff -> +inf)

    # Uniform pseudo-count value for frequencies computation: fi(a) <- (1-theta) fi(a) + theta/q
    # Only used to initialize DCA coefficients, so its impact is very limiter
    theta_regularization = 0.10,

    exclude_gaps = True, # exclude gaps from the DCA model
    min_seqid = 0.25,    # sequences whose sequence-identity with target sequence is below this threshold will be discarded

    # To address redundancy in MSAs, each sequence is reweighted by the inverse of the number of sequences sharing a sequence identity higher than this threshold:
    weights_seqid = 0.80,

    # pLDDT filter parameters:
    use_contacts_plddt_filter = False,
    contacts_plddt_cutoff = 70.0,
    contacts_plddt_keep_window = 1,

    contacts_gap_cutoff = None, # remove contacts at positions where gap-ratio is higher than this threshold (set None to ignore)
    
    count_target_sequence = True, # don't exclude the target sequence from the MSA when inferring the model (may be set to False for mutants for example)
    
    num_threads = 4, # number of threads for parallelization (capped to the number of CPU cores)

    # Cache path parameters:
    # when set to a path, they make it possible to store computed values (when the file does not exist) or retrieve them (when the file exists) from previous runs to avoid unnecessary computations.
    # data is stored in the numpy compressed format and paths must end with .npz
    distance_cache_path = None, # path to store the distance matrix
    rsa_cache_path = None,      # path to store the RSA
    weights_cache_path = None,  # path to store the sequence weights
    dca_cache_path = None,      # path to store the DCA coefficients

    # parameters to control verbosity:
    verbose = True,
    disable_warnings = False,
)